In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [3]:
load_dotenv()
model = ChatOpenAI(model='gpt-5.6-luna')

In [12]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    score: int

In [5]:
def create_outline(state: BlogState) -> BlogState:

    ##fetch title
    title = state['title']

    ##prompt
    prompt = f"Generate a detailed outline for a blog on the topic - {title}"
    outline = model.invoke(prompt).content

    ##update state
    state['outline'] = outline

    return state

In [6]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a blog on the topic {title}, use the following outline \n {outline}'

    blog = model.invoke(prompt).content

    state['content'] = blog

    return state


In [13]:
def eval_blog(state: BlogState) -> BlogState:

    outline = state['outline']
    blog = state['content']

    prompt = f'Based on the outline, rate the generated blog out of 10(give an integer rating). Outline: \n {outline}. Blog: \n {blog}'

    state['score'] = model.invoke(prompt).content

    return state

In [15]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', eval_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [16]:
initial_state = {'title': 'Evolution of the tech-job market in India'}
final_state = workflow.invoke(initial_state)

In [17]:
print(final_state)

{'title': 'Evolution of the tech-job market in India', 'outline': '# Detailed Blog Outline: Evolution of the Tech-Job Market in India\n\n## Working Title  \n**The Evolution of the Tech-Job Market in India: From IT Services to AI, GCCs and the Future of Work**\n\n## Alternative Titles\n- **How India’s Tech-Job Market Has Transformed Over the Last Three Decades**\n- **From Outsourcing to Innovation: The Changing Face of Tech Careers in India**\n- **India’s Tech Employment Story: Trends, Skills and Opportunities in 2026**\n- **The Future of Tech Jobs in India: Roles, Regions and Skills That Matter**\n\n## Primary Objective\n\nExplain how India’s technology employment landscape has evolved—from early software exports and IT services to startups, global capability centres, deep tech, artificial intelligence and flexible work models.\n\n## Target Audience\n\n- Students and fresh graduates exploring technology careers  \n- Mid-career professionals considering reskilling  \n- Recruiters and HR

In [18]:
print(final_state['outline'])

# Detailed Blog Outline: Evolution of the Tech-Job Market in India

## Working Title  
**The Evolution of the Tech-Job Market in India: From IT Services to AI, GCCs and the Future of Work**

## Alternative Titles
- **How India’s Tech-Job Market Has Transformed Over the Last Three Decades**
- **From Outsourcing to Innovation: The Changing Face of Tech Careers in India**
- **India’s Tech Employment Story: Trends, Skills and Opportunities in 2026**
- **The Future of Tech Jobs in India: Roles, Regions and Skills That Matter**

## Primary Objective

Explain how India’s technology employment landscape has evolved—from early software exports and IT services to startups, global capability centres, deep tech, artificial intelligence and flexible work models.

## Target Audience

- Students and fresh graduates exploring technology careers  
- Mid-career professionals considering reskilling  
- Recruiters and HR leaders  
- Entrepreneurs and startup founders  
- Policymakers and education provide

In [19]:
print(final_state['content'])

# The Evolution of the Tech-Job Market in India: From IT Services to AI, GCCs and the Future of Work

**Meta description:** Explore how India’s tech-job market evolved from software outsourcing to startups, GCCs, cloud, AI and specialised technology careers.

India’s technology workforce was once associated mainly with large IT-services companies, software maintenance and business-process outsourcing. Today, the country’s tech-job market includes artificial intelligence researchers, cloud architects, cybersecurity professionals, product engineers, semiconductor designers, data scientists, startup operators and specialists working for global innovation centres.

This transformation has taken place over several decades. India has moved from being primarily an outsourcing destination to becoming an important technology, product and research hub. The industry continues to create opportunities, but the nature of those opportunities is changing.

The central shift is clear: **India’s tech-jo

In [20]:
print(final_state['score'])

9/10
